# Pre- vs. post-SHINE cohort comparison: similarity structure

Do the pre-SHINE and post-SHINE cohorts induce the *same perceptual similarity structure* -- i.e.
does SHINE preprocessing change not just how subjects behave, but what similarity geometry their
judgements imply? Compares the two cohorts' behavioral RDMs directly and through weighted-SMACOF
MDS embeddings, for **retained subjects only** (production cohort, `status == "full data"`,
`task_version >= 4.0`). Companion notebooks: `cohort_comparison.ipynb` (do they behave the same?) and
`cohort_data_quality.ipynb` (is the data equally good?). Figure helpers live in
`analysis/behavioral/cohort_structure.py`.

Each cohort's full 725x725 behavioral RDM is built via `analysis.rdms.behavioral.build_behavioral_rdm()`
(`aggregation="simple"`, the project's validated default per `aggregation_comparison.ipynb`) -- a condensed
distance vector with NaN for the ~35-37% of pairs no retained subject in that cohort ever judged.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from analysis.utils.stats import masked_rank_correlation
from analysis.rdms.behavioral import build_behavioral_rdm
from analysis.rdms.mds import classical_mds_eigenspectrum, effective_rank, run_mds_sweep
from SpAM_Simulations.core.helpers import procrustes_m2
from analysis.behavioral.cohort_comparison import COHORTS, COHORT_LABEL
from analysis.behavioral.cohort_structure import (
    MDS_CACHE_DIR, fig_eigenvalue_scree, fig_stress_scree, fig_cross_cohort_agreement,
)

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in COHORTS}

[behavioral] variant='pre_shine': 79 participants.


[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)


[behavioral] variant='post_shine': 76 participants.


[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 262450)


## 1. Structural comparison via MDS

### 1.1 Eigenvalue scree (classical MDS, raw RDM)

**Caveat -- read before interpreting this plot.** Classical (Torgerson) MDS eigenvalues require a
*complete* distance matrix, so the ~36% of missing pairs are mean-imputed first. This repo's own
`SpAM_Simulations/empirical/gt_construction.py` measured that this is invalid at this exact
coverage level: mean-imputed pairs are mutually equidistant, which manufactures rank rather than
revealing it (a synthetic **rank-8** ground truth put through the identical mask+fill reported an
**effective rank of ~193**, statistically indistinguishable from noise). Our own numbers below land
in the same regime. This plot is kept as a literal, qualitative view per the original ask -- it is
**not** a validated dimensionality estimate. Section 1.3's SMACOF stress scree (no imputation) is
the project's validated alternative.

In [3]:
MAX_DIMS = 15
MDS_DIMS = list(range(2, MAX_DIMS + 1))

eigenvalues = {c: classical_mds_eigenspectrum(rdm[c]) for c in COHORTS}
for c in COHORTS:
    print(f"  {COHORT_LABEL[c]}: effective rank (mean-imputed classical MDS) = {effective_rank(eigenvalues[c])} of 725 -- see caveat above")

fig_eigenvalue_scree(eigenvalues, MAX_DIMS).show()

  Pre-SHINE: effective rank (mean-imputed classical MDS) = 354 of 725 -- see caveat above
  Post-SHINE: effective rank (mean-imputed classical MDS) = 353 of 725 -- see caveat above


### 1.2 Weighted-SMACOF embeddings (R `smacof` backend)

Metric SMACOF via R's `smacof::mds()` (`SpAM_Simulations/core/multi_dimensional_scaling.py`'s
`run_mds()`), with binary observed/unobserved weights -- unlike classical MDS, missing pairs are
simply excluded from the fit, no imputation. One fit per dimensionality 2..`MAX_DIMS` per cohort
(28 fits total); each real-scale fit takes ~15-30s, so results are cached to
`analysis/results/behavioral/mds_cache/` (one `.npz` per cohort x dimension) and re-runs of this
notebook reuse the cache instead of refitting.

In [4]:
mds_results = {
    c: run_mds_sweep(rdm[c], MDS_DIMS, c, MDS_CACHE_DIR, max_iters=1000, convergence_tol=1e-6)
    for c in COHORTS
}
for c in COHORTS:
    print(f"  {COHORT_LABEL[c]}: {len(mds_results[c])} dimensionalities ready (cached under {MDS_CACHE_DIR})")

  Pre-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)
  Post-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)


### 1.3 Stress scree plot

Weighted-SMACOF stress-1 across dimensionality, one trace per cohort -- the validated
(no-imputation) dimensionality diagnostic, in contrast to section 1.1's eigenvalue scree.

In [5]:
fig_stress_scree(mds_results, MDS_DIMS).show()

### 1.4 Pre- vs. post-SHINE structural agreement

For the raw RDMs and for each embedding dimensionality, do the pre- and post-SHINE cohorts agree
on the *same* similarity structure? Kendall tau and Spearman rho are computed on the raw RDMs too
(restricted to jointly-covered pairs), but Procrustes m^2 needs a complete point configuration, so
it's only defined once an MDS embedding exists (`NaN` for "raw RDM" below). Procrustes m^2 is
shown as `1 - m^2` so all three traces read "bigger = better".

In [6]:
agreement: dict[str, dict] = {}
agreement["raw RDM"] = {**masked_rank_correlation(rdm["pre"], rdm["post"]), "procrustes_m2": None}
for d in MDS_DIMS:
    ks = masked_rank_correlation(mds_results["pre"][d]["confdist"], mds_results["post"][d]["confdist"])
    m2 = procrustes_m2(mds_results["pre"][d]["conf"], mds_results["post"][d]["conf"])
    agreement[str(d)] = {**ks, "procrustes_m2": m2}

agreement_table = pd.DataFrame(agreement).T[["kendall", "spearman", "procrustes_m2", "n_overlap"]]
agreement_table

,kendall,spearman,procrustes_m2,n_overlap
raw RDM,0.147864,0.219826,NaN,107040.0
2,0.502793,0.689974,0.305794,262450.0
3,0.435604,0.613894,0.375806,262450.0
4,0.404671,0.573814,0.442914,262450.0
5,0.406591,0.574581,0.477917,262450.0
6,0.402729,0.567673,0.550900,262450.0
7,0.418474,0.586165,0.509305,262450.0
8,0.429431,0.599686,0.527540,262450.0
9,0.436127,0.606954,0.534334,262450.0
10,0.447335,0.620495,0.535048,262450.0


In [7]:
fig_cross_cohort_agreement(agreement, MDS_DIMS).show()